In [1]:
import pickle, json, sys, os
import numpy as np
import pandas as pd
import spacy
from IPython.display import HTML, display

sys.path.insert(0, '/code/HLTproject_code/task_1B_complexity_explanation')
from utils.parse_responses import extract_annotation

nlp = spacy.load('it_core_news_md')

In [2]:
# ── carica eval_set e risposte ANITA ──────────────────────────────────────────
with open('/code/HLTproject_code/task_1B_complexity_explanation/data/eval_set_full.pkl', 'rb') as f:
    eval_set = pickle.load(f)

with open('/code/HLTproject_code/task_1B_complexity_explanation/anita_results/anita_resp_full.pkl', 'rb') as f:
    anita_raw = pickle.load(f)

print(f'eval_set: {len(eval_set)} frasi')
print(f'anita_raw: {len(anita_raw)} risposte')

eval_set: 7539 frasi
anita_raw: 7539 risposte


In [3]:
import numpy as np
rate = np.mean([np.mean(ex["silver"]) for ex in eval_set])
print(f"silver positive rate: {rate:.3f}") 

silver positive rate: 0.339


In [4]:
# ── baseline random ───────────────────────────────────────────────────────────
rng = np.random.default_rng(42)

def predict_random(eval_set, rate=0.34):
    preds = []
    for ex in eval_set:
        n = len(ex['tokens'])
        k = int(round(rate * n))
        idx = rng.choice(n, size=min(k, n), replace=False)
        p = [0] * n
        for i in idx:
            p[i] = 1
        preds.append(p)
    return preds

preds_random = predict_random(eval_set)
print('random OK')

random OK


In [5]:
# ── baseline VdB ─────────────────────────────────────────────────────────────
def carica_vdb(path):
    vdb = set()
    with open(path, encoding='utf-8') as f:
        for line in f:
            w = line.strip().split('\t')[0].split(',')[0].lower()
            if w and not w.startswith('#'):
                vdb.add(w)
    return vdb

CONTENT = {'NOUN', 'VERB', 'ADJ', 'ADV'}

def predict_vdb(eval_set, vdb):
    preds = []
    for ex in eval_set:
        p = [int(pos in CONTENT and lem not in vdb)
             for lem, pos in zip(ex['lemmas'], ex['pos'])]
        preds.append(p)
    return preds

vdb = carica_vdb('/code/HLTproject_code/nvdb/nvdb.words.txt')
preds_vdb = predict_vdb(eval_set, vdb)
print(f'VdB: {len(vdb)} lemmi, predizioni OK')

VdB: 7181 lemmi, predizioni OK


In [6]:
# ── ANITA: da risposta LLM → vettore binario ─────────────────────────────────
def anita_to_binary(eval_set, anita_raw):
    preds, stats = [], {}
    for ex, raw in zip(eval_set, anita_raw):
        n = len(ex['tokens'])
        parsed, status = extract_annotation(raw, n_tokens=n)
        stats[status] = stats.get(status, 0) + 1
        p = [0] * n
        if parsed:
            for idx in parsed['complex']:
                if 0 <= idx < n:
                    p[idx] = 1
        preds.append(p)
    print('Parsing ANITA:', stats)
    return preds

preds_anita = anita_to_binary(eval_set, anita_raw)

Parsing ANITA: {'recovered_regex': 2760, 'ok': 4779}


In [7]:
np.mean([np.mean(p) for p in preds_anita]) 

np.float64(0.12802259767968438)

In [7]:
# ── IG: carica da file se disponibile, altrimenti ricalcola su subset ─────────
IG_PREDS_PATH = '/code/HLTproject_code/task_1B_complexity_explanation/preds/preds_ig.pkl'

if os.path.exists(IG_PREDS_PATH):
    with open(IG_PREDS_PATH, 'rb') as f:
        preds_ig = pickle.load(f)
    print(f'IG predizioni caricate da file: {len(preds_ig)}')
else:
    print('preds_ig.pkl non trovato — esegui il notebook integrated_gradients_metrics.ipynb')
    print('e aggiungi alla fine: pickle.dump(preds_ig, open("preds_ig.pkl","wb"))')
    preds_ig = None

IG predizioni caricate da file: 7539


In [8]:
PALETTE = {
    'silver': '#90EE90',   # verde
    'vdb':    '#ADD8E6',   # azzurro
    'random': '#FFD700',   # giallo
    'ig':     '#FA8072',   # salmone
    'anita':  '#DDA0DD',   # viola
}

def highlight_sentence(tokens, binary, color):
    parts = []
    for t, b in zip(tokens, binary):
        if b:
            parts.append(
                f'<span style="background:{color};padding:1px 4px;'
                f'border-radius:3px;margin:1px;display:inline-block">{t}</span>'
            )
        else:
            parts.append(f'<span style="margin:1px;display:inline-block">{t}</span>')
    return ' '.join(parts)


def show_examples(eval_set, preds_dict, indices):
    """
    preds_dict = {'vdb': preds_vdb, 'random': preds_random, 'ig': preds_ig, 'anita': preds_anita}
    indices    = lista di indici in eval_set da visualizzare
    """
    html = []
    for i in indices:
        ex = eval_set[i]
        toks = ex['tokens']
        html.append(f'<div style="border:1px solid #ccc;padding:10px;margin:10px 0;border-radius:6px">')
        html.append(f'<b>Esempio {i}</b> (id={ex["idx"]}) &nbsp;&nbsp; '
                    f'<small>({len(toks)} token, '
                    f'{sum(ex["silver"])} complessi secondo silver)</small><br><br>')
        html.append(f'<b>Originale:</b> {" ".join(toks)}<br>')
        html.append(f'<b>Semplificata:</b> <i>{" ".join(ex["simp_tokens"])}</i><br><br>')

        # silver
        html.append(f'<b style="color:#2d7a2d">Silver:</b>&nbsp;&nbsp;'
                    + highlight_sentence(toks, ex['silver'], PALETTE['silver']) + '<br>')

        # metodi
        for method, preds in preds_dict.items():
            if preds is None:
                html.append(f'<b>{method.upper()}:</b>&nbsp;&nbsp;<i>non disponibile</i><br>')
                continue
            p = preds[i]
            n_ev = sum(p)
            html.append(
                f'<b style="color:#333">{method.upper()}:</b>&nbsp;&nbsp;'
                + highlight_sentence(toks, p, PALETTE[method])
                + f'&nbsp;<small>({n_ev}/{len(toks)})</small><br>'
            )

        html.append('</div>')
    display(HTML(''.join(html)))

In [9]:
def _latex_escape(s):
    """Escape dei caratteri speciali LaTeX in un token."""
    replacements = {
        '\\': r'\textbackslash{}',
        '&': r'\&', '%': r'\%', '$': r'\$', '#': r'\#',
        '_': r'\_', '{': r'\{', '}': r'\}',
        '~': r'\textasciitilde{}', '^': r'\textasciicircum{}',
    }
    return ''.join(replacements.get(c, c) for c in s)


def _hex_to_rgb01(hex_color):
    hex_color = hex_color.lstrip('#')
    r = int(hex_color[0:2], 16) / 255
    g = int(hex_color[2:4], 16) / 255
    b = int(hex_color[4:6], 16) / 255
    return f"{r:.3f},{g:.3f},{b:.3f}"


def _highlight_sentence_latex(tokens, binary, color_name):
    parts = []
    for t, b in zip(tokens, binary):
        t_esc = _latex_escape(t)
        parts.append(f"\\colorbox{{{color_name}}}{{{t_esc}}}" if b else t_esc)
    return " ".join(parts)


def examples_to_latex_table(eval_set, preds_dict, indices, method_order=None, col_width="11cm"):
    """
    Genera una longtable LaTeX con UN esempio per riga: colonna ID + colonna
    con Originale, Silver, ogni metodo e Semplificata impilati (a capo con
    \\newline), evidenziati con lo stesso schema colori di PALETTE.

    Richiede nel preambolo: \\usepackage{xcolor}, \\usepackage{longtable}, \\usepackage{booktabs}

    preds_dict = {'vdb': preds_vdb, 'random': preds_random, 'ig': preds_ig, 'anita': preds_anita}
    indices    = lista di indici in eval_set da esportare (es. list(range(10)) o indici a mano)
    """
    if method_order is None:
        method_order = [m for m in preds_dict if preds_dict[m] is not None]

    color_defs = "\n".join(
        f"\\definecolor{{col{name}}}{{rgb}}{{{_hex_to_rgb01(hexcode)}}}"
        for name, hexcode in PALETTE.items()
    )

    header = (
        f"\\begin{{longtable}}{{@{{}} c p{{{col_width}}} @{{}}}}\n"
        "\\toprule\n"
        "\\textbf{ID} & \\textbf{Confronto} \\\\\n"
        "\\midrule\n"
        "\\endhead\n"
    )
    footer = "\\bottomrule\n\\end{longtable}"

    rows = []
    for i in indices:
        ex = eval_set[i]
        toks = ex['tokens']

        cell_lines = [
            r"\textbf{Originale:} " + _latex_escape(' '.join(toks)),
            r"\textbf{Silver:} " + _highlight_sentence_latex(toks, ex['silver'], 'colsilver'),
        ]
        for method in method_order:
            preds = preds_dict.get(method)
            if preds is None:
                continue
            p = preds[i]
            cell_lines.append(
                f"\\textbf{{{method.upper()}:}} " + _highlight_sentence_latex(toks, p, f'col{method}')
            )
        cell_lines.append(
            r"\textbf{Semplificata:} " + _latex_escape(' '.join(ex['simp_tokens']))
        )

        cell = " \\newline\n".join(cell_lines)
        rows.append(f"{i} &\n{cell} \\\\ \\midrule")

    body = "\n".join(rows)
    return color_defs + "\n\n" + header + body + "\n" + footer


In [22]:
# ── scegli gli esempi da visualizzare ─────────────────────────────────────────
# puoi mettere indici specifici o usare range(N)
INDICES = list(range(7))

preds_dict = {
    'vdb':    preds_vdb,
    'random': preds_random,
    'ig':     preds_ig,    # None se non caricato
    'anita':  preds_anita,
}

show_examples(eval_set, preds_dict, INDICES)

In [11]:
def find_disagreements(eval_set, preds_dict, top_k=10):
    """
    Cerca le frasi dove le predizioni dei metodi disponibili divergono di più.
    Misura: deviazione standard media della probabilità di evidenziazione per token.
    """
    available = {k: v for k, v in preds_dict.items() if v is not None}
    if len(available) < 2:
        print('Servono almeno 2 metodi disponibili.')
        return []

    scores = []
    for i, ex in enumerate(eval_set):
        n = len(ex['tokens'])
        mat = np.array([available[k][i][:n] for k in available])  # (n_methods, n_tok)
        score = mat.std(axis=0).mean()   # disaccordo medio per token
        scores.append((score, i))

    scores.sort(reverse=True)
    return [i for _, i in scores[:top_k]]


disagreement_indices = find_disagreements(eval_set, preds_dict, top_k=3)
print('Top-3 esempi con più disaccordo tra i metodi:', disagreement_indices)

Top-3 esempi con più disaccordo tra i metodi: [4653, 6870, 356]


In [12]:
show_examples(eval_set, preds_dict, disagreement_indices)

In [23]:
INDICES = list(range(3))   # oppure una lista di indici a piacere, es. [10, 42, 215, ...]
INDICES = [3]

latex_code = examples_to_latex_table(eval_set, preds_dict, INDICES)
print(latex_code)

\definecolor{colsilver}{rgb}{0.565,0.933,0.565}
\definecolor{colvdb}{rgb}{0.678,0.847,0.902}
\definecolor{colrandom}{rgb}{1.000,0.843,0.000}
\definecolor{colig}{rgb}{0.980,0.502,0.447}
\definecolor{colanita}{rgb}{0.867,0.627,0.867}

\begin{longtable}{@{} c p{11cm} @{}}
\toprule
\textbf{ID} & \textbf{Confronto} \\
\midrule
\endhead
3 &
\textbf{Originale:} Pur di riconquistare la ex ragazza Elaine ora impiegata come hostess , Striker decide di trovare il coraggio , imbarcandosi in un volo per Chicago dove ella presta servizio . \newline
\textbf{Silver:} \colorbox{colsilver}{Pur} \colorbox{colsilver}{di} riconquistare \colorbox{colsilver}{la} \colorbox{colsilver}{ex} \colorbox{colsilver}{ragazza} Elaine \colorbox{colsilver}{ora} \colorbox{colsilver}{impiegata} \colorbox{colsilver}{come} \colorbox{colsilver}{hostess} , Striker \colorbox{colsilver}{decide} \colorbox{colsilver}{di} \colorbox{colsilver}{trovare} \colorbox{colsilver}{il} \colorbox{colsilver}{coraggio} , \colorbox{colsilver}{im

In [14]:
#with open('examples_table.tex', 'w', encoding='utf-8') as f:
  #  f.write(latex_code)

In [15]:
import sys
sys.path.append("/code/HLTproject_code/task_1B_complexity_explanation/baselines")

from explanations import evaluate, silver_labels , spiega_token , delta_profiling

/data/miniconda3/envs/hlt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
import sys
sys.path.insert(0, '/code/HLTproject_code/task_1B_complexity_explanation/baselines')
from explanations import spiega_token, delta_profiling, carica_vdb

vdb = carica_vdb('/code/HLTproject_code/nvdb/nvdb.words.txt')  # se non già caricato

# ── spiega_token su UNA frase specifica ──────────────────────────────────────
i = 0
ex = dict(eval_set[i])  # copia per non modificare l'eval_set originale

# aggiungo "dep" (mancante in eval_set_full.pkl), stesso procedimento usato per gli altri campi
doc = [t for t in nlp(ex["original"]) if not t.is_space]
ex["dep"] = [t.dep_ for t in doc]
assert len(ex["dep"]) == len(ex["tokens"])   # controllo di allineamento

spiegazioni = spiega_token(ex, vdb)
for s in spiegazioni:
    print(s)   # {'token': ..., 'operazione': 'del'|'sub', 'motivo': '...'}

{'token': 'Prodotto', 'operazione': 'sub', 'motivo': 'clausola subordinata/participiale (sintattico)'}
{'token': 'dalla', 'operazione': 'sub', 'motivo': 'altro'}
{'token': 'BBC', 'operazione': 'sub', 'motivo': 'altro'}
{'token': 'solo', 'operazione': 'sub', 'motivo': 'altro'}
{'token': 'ottiene', 'operazione': 'sub', 'motivo': 'altro'}
{'token': 'candidatura', 'operazione': 'sub', 'motivo': 'parola non di base (lessicale)'}
{'token': 'Premio', 'operazione': 'sub', 'motivo': 'altro'}
{'token': 'Oscar', 'operazione': 'sub', 'motivo': 'altro'}
{'token': 'per', 'operazione': 'sub', 'motivo': 'altro'}
{'token': 'miglior', 'operazione': 'sub', 'motivo': 'altro'}
{'token': 'cortometraggio', 'operazione': 'sub', 'motivo': 'parola non di base (lessicale)'}


In [17]:
import pandas as pd

test = pd.read_parquet('/code/HLTproject_code/data/test.parquet')
dz = delta_profiling(test)   # una volta sola per tutto il test set (DataFrame z-normalizzato)

# ── su UNA frase specifica ───────────────────────────────────────────────────
i = 0
ex = eval_set[i]
print(f"Frase idx={ex['idx']}: {ex['original']}")
print(f"Dimensioni ProfilingUD ridotte di più nella frase idx={ex['idx']}:")
print(dz.loc[ex['idx']].sort_values(ascending=False).head(5))

Frase idx=10: Prodotto dalla BBC, il film esce solo nel 1998 ed ottiene numerosi riconoscimenti internazionali, tra cui la candidatura al Premio Oscar per il miglior cortometraggio animato.
Dimensioni ProfilingUD ridotte di più nella frase idx=10:
subordinate_pre               3.215672
dep_dist_obl:agent            1.828635
verbs_tense_dist_Pres         1.588690
verbs_num_pers_dist_Sing+3    1.565904
subordinate_dist_1            1.461724
Name: 10, dtype: float64


In [18]:
from sklearn.metrics import precision_recall_fscore_support
from sacrebleu.metrics import TER

rows = []
for method, preds in preds_dict.items():
    if preds is None:
        continue
    m = evaluate(eval_set, preds)
    rows.append({'metodo': method, **{k: round(v, 3) for k, v in m.items()}})

pd.DataFrame(rows).set_index('metodo')

OK: tutte 7539 le frasi sono allineate (tokens == silver == pred).
OK: tutte 7539 le frasi sono allineate (tokens == silver == pred).
OK: tutte 7539 le frasi sono allineate (tokens == silver == pred).
OK: tutte 7539 le frasi sono allineate (tokens == silver == pred).


,precision,recall,f1,ed_sub1,ed_sub1.5,ed_sub2,ter,n_valutate
metodo,,,,,,,,
vdb,0.347,0.096,0.129,20.016,22.961,25.414,87.943,7539
random,0.338,0.319,0.299,19.800,22.900,25.363,81.534,7539
ig,0.392,0.404,0.358,19.371,22.189,24.447,79.075,7539
anita,0.385,0.137,0.180,19.890,22.870,25.333,85.565,7539


In [19]:
vuote = sum(1 for p in preds_anita if sum(p) == 0)
print(f"frasi senza highlight LLM: {vuote}/{len(preds_anita)}")

frasi senza highlight LLM: 123/7539
